# MIA-203 · Examen final
## Detección de minería artesanal y otras clases raras en imágenes satelitales de la Amazonía

Este notebook contiene todo el proyecto, de principio a fin: datos, partición, línea base, funciones de
pérdida, entrenamiento de ResNet-50 y DeiT-Small (ViT), tablas, figuras e interpretabilidad (Grad-CAM).

**Versión oficial.** Los números de las celdas de texto corresponden a la **versión 4** del notebook en
Kaggle, ejecutada completa desde cero con *Save & Run All* (GPU Tesla T4, unas 2 horas). Son los mismos
que reportamos en el artículo.

**Sobre la reproducibilidad.** Activamos `cudnn.benchmark` para acelerar las convoluciones. Por eso, al
volver a ejecutar el notebook, los resultados de ResNet-50 pueden variar ligeramente (en la ejecución de
desarrollo, la media de la mAP de raras cambió entre 0,004 y 0,025 según la pérdida, sin alterar el orden
de las pérdidas). Los resultados de DeiT-Small y de la línea base se reprodujeron exactamente.

**Cómo ejecutarlo.** En Kaggle, con GPU T4, Internet activado y como datos de entrada la competencia
*Planet: Understanding the Amazon from Space* (etiquetas) y el dataset *nikitarom/planets-dataset*
(imágenes). Las celdas están pensadas para ejecutarse en orden; todo lo generado se guarda en
`/kaggle/working`.

In [ ]:
# Verificamos qué datos quedaron conectados al notebook
# Vamos a repetir la invocación a las librerias para evitar retroceder al día siguiente
import os
from pathlib import Path

RAIZ = Path('/kaggle/input')  # Kaggle monta aquí todos los datos de "Input"

# Recorremos las carpetas de entrada y mostramos su estructura (hasta 3 niveles)
for ruta, carpetas, archivos in os.walk(RAIZ):
    nivel = len(Path(ruta).relative_to(RAIZ).parts)
    if nivel >= 3:
        carpetas[:] = []  # no bajamos más para no imprimir demasiado
    sangria = '    ' * nivel
    # Contamos cuántas imágenes JPG hay directamente en esta carpeta
    n_jpg = sum(1 for a in archivos if a.lower().endswith('.jpg'))
    print(f'{sangria}{Path(ruta).name}/  ({len(archivos)} archivos, {n_jpg} jpg)')
    # Mostramos los archivos que no son imágenes (CSV, torrent, etc.)
    for a in archivos:
        if not a.lower().endswith('.jpg'):
            print(f'{sangria}    - {a}')

# Validación: buscamos las carpetas de imágenes de entrenamiento y contamos sus archivos
carpetas_train = [p for p in RAIZ.rglob('train-jpg') if p.is_dir()]
print('\nCarpetas train-jpg encontradas:', len(carpetas_train))
for d in carpetas_train:
    print(f'  {d} -> {sum(1 for _ in d.glob("*.jpg"))} imágenes')

In [ ]:
# =====================================================================
# BLOQUE 1 · Entorno, rutas, carga de etiquetas y análisis exploratorio
# =====================================================================

# --- 1.1 Importamos las librerías que usaremos en este bloque ---
import os, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision, sklearn

# Verificamos que timm (librería de modelos preentrenados) esté instalada; si no, la instalamos
try:
    import timm
except ImportError:
    os.system('pip install -q timm==1.0.9')
    import timm

# --- 1.2 Definimos las rutas de entrada y de trabajo ---
ENTRADA = Path('/kaggle/input')     # aquí Kaggle monta los datos de solo lectura
TRABAJO = Path('/kaggle/working')   # aquí guardamos todo lo que generamos (persiste con "Files only")

# Localizamos la carpeta de imágenes y el CSV oficial de etiquetas sin escribir las rutas a mano
DIR_IMG = next(p for p in ENTRADA.rglob('train-jpg') if p.is_dir())
CSV_ETIQ = next(p for p in ENTRADA.rglob('train_v2.csv') if p.is_file() and 'competitions' in p.parts)

# Creamos las carpetas donde guardaremos particiones, resultados, figuras, registros y pesos
for sub in ['splits', 'results/tables', 'results/figures', 'results/logs', 'results/preds', 'weights']:
    (TRABAJO / sub).mkdir(parents=True, exist_ok=True)

# --- 1.3 Registramos el entorno (para requirements.txt y para el reporte de costo, R6) ---
entorno = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'timm': timm.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU',
    'n_gpus': torch.cuda.device_count(),
}
with open(TRABAJO / 'results/entorno.json', 'w') as f:
    json.dump(entorno, f, indent=1)
print('ENTORNO:', json.dumps(entorno, indent=1))

# --- 1.4 Cargamos las etiquetas oficiales y comprobamos que cada una tenga su imagen ---
df = pd.read_csv(CSV_ETIQ)                       # columnas: image_name, tags
nombres_disco = {f[:-4] for f in os.listdir(DIR_IMG) if f.endswith('.jpg')}
faltan = set(df.image_name) - nombres_disco      # etiquetas sin imagen
sobran = nombres_disco - set(df.image_name)      # imágenes sin etiqueta

# Convertimos el texto de etiquetas ("clear primary water") en una matriz binaria (una columna por etiqueta)
df['lista'] = df.tags.str.split()
ETIQUETAS = sorted({t for lista in df.lista for t in lista})
Y = np.array([[int(t in set(lista)) for t in ETIQUETAS] for lista in df.lista], dtype=np.uint8)
df_bin = pd.concat([df[['image_name']], pd.DataFrame(Y, columns=ETIQUETAS)], axis=1)
df_bin.to_csv(TRABAJO / 'splits/etiquetas_binarias.csv', index=False)  # la reutilizamos en los siguientes bloques

# --- 1.5 Contamos cuántas imágenes tiene cada etiqueta (análisis exploratorio) ---
conteo = pd.DataFrame({'etiqueta': ETIQUETAS, 'imagenes': Y.sum(0)})
conteo['porcentaje'] = 100 * conteo.imagenes / len(df)
conteo = conteo.sort_values('imagenes', ascending=False).reset_index(drop=True)

# Definimos como "raras" las etiquetas presentes en menos del 1,5 % de las imágenes
UMBRAL_RARA = 1.5
RARAS = conteo.loc[conteo.porcentaje < UMBRAL_RARA, 'etiqueta'].tolist()
conteo['rara'] = conteo.etiqueta.isin(RARAS)
conteo.to_csv(TRABAJO / 'results/tables/tabla1_etiquetas.csv', index=False)

# Graficamos la distribución en escala logarítmica, resaltando en rojo las etiquetas raras
fig, ax = plt.subplots(figsize=(8, 5))
colores = ['tab:red' if r else 'tab:blue' for r in conteo.rara]
ax.barh(conteo.etiqueta, conteo.imagenes, color=colores)
ax.set_xscale('log'); ax.invert_yaxis()
ax.set_xlabel('Número de imágenes (escala logarítmica)'); ax.set_ylabel('Etiqueta')
ax.set_title(f'Distribución de etiquetas (rojo: < {UMBRAL_RARA} % de las imágenes)')
plt.tight_layout(); plt.savefig(TRABAJO / 'results/figures/fig1_distribucion_etiquetas.png', dpi=200); plt.show()

# Vemos con qué otras etiquetas aparece la minería artesanal (útil para la discusión del paper)
mina = df_bin[df_bin.artisinal_mine == 1]
coocurrencia = (mina[ETIQUETAS].mean() * 100).drop('artisinal_mine').sort_values(ascending=False).round(1)
coocurrencia.to_csv(TRABAJO / 'results/tables/tabla_coocurrencia_mineria.csv', header=['porcentaje'])

# Guardamos la configuración global para que los siguientes bloques (y otros días) la reutilicen
config = {'DIR_IMG': str(DIR_IMG), 'CSV_ETIQ': str(CSV_ETIQ), 'ETIQUETAS': ETIQUETAS,
          'RARAS': RARAS, 'UMBRAL_RARA': UMBRAL_RARA}
with open(TRABAJO / 'config_global.json', 'w') as f:
    json.dump(config, f, indent=1)

# --- 1.6 Validación: si algo no cuadra, nos detenemos con un mensaje claro ---
assert len(df) == 40479, f'Esperábamos 40.479 filas y hay {len(df)}'
assert not faltan, f'Hay {len(faltan)} etiquetas sin imagen'
assert len(ETIQUETAS) == 17, f'Esperábamos 17 etiquetas y hay {len(ETIQUETAS)}'
assert Y.sum(1).min() >= 1, 'Hay imágenes sin ninguna etiqueta'

print('\n===== RESUMEN BLOQUE 1 =====')
print('Imágenes:', len(df), '| etiquetas sin imagen:', len(faltan), '| imágenes sin etiqueta:', len(sobran))
print('Etiquetas por imagen (promedio):', round(Y.sum(1).mean(), 2))
print('Etiquetas raras:', RARAS)
print('\nConteo por etiqueta:\n', conteo.to_string(index=False))
print('\nEtiquetas que acompañan a artisinal_mine (% de sus imágenes):\n', coocurrencia.head(8).to_string())
print('\nBLOQUE 1 OK ✔')

## Resultados del Bloque 1 · Entorno, datos y análisis exploratorio

### Qué hicimos
Registramos el entorno de ejecución, cargamos las etiquetas oficiales de la competencia
(`train_v2.csv`) y verificamos que cada etiqueta tuviera su imagen correspondiente.
Luego contamos la frecuencia de cada una de las 17 etiquetas y analizamos con qué otras
etiquetas aparece la minería artesanal.

**Entorno:** Kaggle con GPU Tesla T4 (usamos 1 de 2), Python 3.13.16, PyTorch 2.11.0,
torchvision 0.26.0, timm 1.0.30, NumPy 2.1.3, pandas 2.3.3, scikit-learn 1.6.1.

**Fuentes de datos:**
- Etiquetas: competencia *Planet: Understanding the Amazon from Space* (Planet Labs y Kaggle, 2017),
  sujetas a las reglas de la competencia.
- Imágenes: copia en Kaggle Datasets *planets-dataset* (usuario nikitarom), licencia ODbL,
  porque la competencia ya no aloja los archivos de imágenes.
- Imágenes JPG de 256×256 píxeles, tomadas entre enero de 2016 y febrero de 2017 en la cuenca
  amazónica (incluido el Perú), con unos 3,7 m por píxel.

### Qué obtuvimos
- **40.479 imágenes**, todas con etiqueta y sin archivos faltantes.
- **17 etiquetas**, con un promedio de **2,87 etiquetas por imagen** (problema multietiqueta).
- Definimos como **raras** las etiquetas presentes en menos del 1,5 % de las imágenes:

| Etiqueta rara | Imágenes | % del total |
|---|---|---|
| selective_logging (tala selectiva) | 340 | 0,84 % |
| artisinal_mine (minería artesanal) | 339 | 0,84 % |
| blooming (floración) | 332 | 0,82 % |
| slash_burn (tala y quema) | 209 | 0,52 % |
| conventional_mine (minería convencional) | 100 | 0,25 % |
| blow_down (árboles derribados por viento) | 98 | 0,24 % |

- Entre las imágenes con **minería artesanal**, el **88,2 % también tiene agua**, el 32,4 %
  carreteras y el 11,8 % suelo desnudo.

### Qué significa
- El desbalance es extremo: la etiqueta más común (`primary`, 92,7 %) es unas 110 veces más
  frecuente que la minería artesanal. Una pérdida estándar tenderá a ignorar las clases raras,
  lo que justifica comparar funciones de pérdida diseñadas para el desbalance (focal loss, ASL).
- La fuerte asociación entre minería y agua es coherente con la minería aluvial, que opera junto
  a ríos y deja pozas. Pero es también un **riesgo de atajo**: el modelo podría aprender
  "agua = mina" en lugar de reconocer la mina. Lo evaluaremos con Grad-CAM (R5).
- Con tan pocos ejemplos, el conjunto de test tendrá unas 50 imágenes de minería artesanal y
  unas 15 de `blow_down` y `conventional_mine`. Por eso la partición debe repartir con cuidado
  las clases raras y reportaremos resultados con 3 semillas (media ± desviación estándar).

**Archivos generados:** `results/entorno.json`, `splits/etiquetas_binarias.csv`,
`results/tables/tabla1_etiquetas.csv`, `results/tables/tabla_coocurrencia_mineria.csv`,
`results/figures/fig1_distribucion_etiquetas.png`, `config_global.json`.

In [ ]:
# =====================================================================
# BLOQUE 2 · Partición train / validación / test (70 / 15 / 15)
# =====================================================================

# --- 2.1 Importamos librerías y recuperamos lo guardado en el bloque 1 ---
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

TRABAJO = Path('/kaggle/working')
config = json.load(open(TRABAJO / 'config_global.json'))       # rutas, etiquetas y etiquetas raras
ETIQUETAS, RARAS = config['ETIQUETAS'], config['RARAS']
df_bin = pd.read_csv(TRABAJO / 'splits/etiquetas_binarias.csv')  # una fila por imagen, una columna por etiqueta

SEMILLA_PARTICION = 42                                  # semilla fija: la partición será siempre la misma
RUTA_PARTICION = TRABAJO / 'splits/particion.csv'

# --- 2.2 Definimos el estrato de cada imagen: su etiqueta más rara ---
# Al estratificar por la etiqueta menos frecuente de cada imagen, garantizamos que las clases raras
# se repartan en la misma proporción en train, validación y test.
frecuencia = df_bin[ETIQUETAS].sum()                    # cuántas imágenes tiene cada etiqueta
orden_rareza = frecuencia.sort_values().index.tolist()  # de la más rara a la más común

def etiqueta_mas_rara(fila):
    # Recorremos las etiquetas de la más rara a la más común y devolvemos la primera presente
    for e in orden_rareza:
        if fila[e] == 1:
            return e

df_bin['estrato'] = df_bin.apply(etiqueta_mas_rara, axis=1)

# Por seguridad, juntamos en un estrato "otros" los que tengan menos de 10 imágenes
tamanos = df_bin.estrato.value_counts()
df_bin.loc[df_bin.estrato.isin(tamanos[tamanos < 10].index), 'estrato'] = 'otros'

# --- 2.3 Hacemos la partición (o la cargamos si ya existe, para no rehacerla nunca) ---
if RUTA_PARTICION.exists():
    particion = pd.read_csv(RUTA_PARTICION)
    print('Partición existente cargada desde', RUTA_PARTICION)
else:
    # Paso 1: separamos 70 % para entrenamiento y 30 % temporal, estratificando
    train_idx, temp_idx = train_test_split(df_bin.index, test_size=0.30,
                                           stratify=df_bin.estrato, random_state=SEMILLA_PARTICION)
    # Paso 2: dividimos el 30 % temporal en dos mitades: validación (15 %) y test (15 %)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50,
                                         stratify=df_bin.loc[temp_idx, 'estrato'],
                                         random_state=SEMILLA_PARTICION)
    particion = pd.DataFrame({'image_name': df_bin.image_name, 'split': ''})
    particion.loc[train_idx, 'split'] = 'train'
    particion.loc[val_idx, 'split'] = 'val'
    particion.loc[test_idx, 'split'] = 'test'
    particion.to_csv(RUTA_PARTICION, index=False)
    print('Partición nueva creada y guardada en', RUTA_PARTICION)

# --- 2.4 Revisamos cuántas imágenes de cada etiqueta quedaron en cada partición ---
datos = df_bin.merge(particion, on='image_name')
tabla = datos.groupby('split')[ETIQUETAS].sum().T[['train', 'val', 'test']]
tabla['total'] = tabla.sum(axis=1)
for s in ['train', 'val', 'test']:
    tabla[f'%{s}'] = (100 * tabla[s] / tabla.total).round(1)   # qué porcentaje de cada etiqueta cayó en cada parte
tabla = tabla.loc[frecuencia.sort_values(ascending=False).index]
tabla.to_csv(TRABAJO / 'results/tables/tabla2_particion.csv')

# --- 2.5 Validación: si algo no cuadra, nos detenemos con un mensaje claro ---
conjuntos = {s: set(particion.image_name[particion.split == s]) for s in ['train', 'val', 'test']}
assert not (conjuntos['train'] & conjuntos['val']), 'Hay imágenes repetidas entre train y val'
assert not (conjuntos['train'] & conjuntos['test']), 'Hay imágenes repetidas entre train y test'
assert not (conjuntos['val'] & conjuntos['test']), 'Hay imágenes repetidas entre val y test'
assert sum(len(c) for c in conjuntos.values()) == len(df_bin), 'Se perdieron imágenes en la partición'
for e in RARAS:
    assert tabla.loc[e, 'val'] >= 10 and tabla.loc[e, 'test'] >= 10, f'{e} tiene muy pocos ejemplos en val o test'

print('\n===== RESUMEN BLOQUE 2 =====')
print('Tamaños:', {s: len(c) for s, c in conjuntos.items()})
print('Proporciones:', {s: f'{100 * len(c) / len(df_bin):.1f} %' for s, c in conjuntos.items()})
print('\nImágenes por etiqueta y partición:\n', tabla.to_string())
print('\nBLOQUE 2 OK ✔')

## Resultados del Bloque 2 · Partición de los datos

### Qué hicimos
Dividimos las 40.479 imágenes en entrenamiento (70 %), validación (15 %) y test (15 %) con una
semilla fija (42). Para que las clases raras quedaran bien repartidas, estratificamos según la
**etiqueta más rara de cada imagen**: si una imagen tiene, por ejemplo, `clear`, `primary`,
`water` y `artisinal_mine`, su estrato es `artisinal_mine`. La partición se guardó en
`splits/particion.csv` y es la **misma para todos los experimentos**.

### Qué obtuvimos
- Tamaños: **28.335 / 6.072 / 6.072** imágenes (70,0 / 15,0 / 15,0 %).
- Las 17 etiquetas quedaron cerca de la proporción 70 / 15 / 15; la mayor desviación fue de
  0,7 puntos porcentuales.
- Etiquetas raras en test: `artisinal_mine` 51, `selective_logging` 51, `blooming` 51,
  `slash_burn` 31, `conventional_mine` 15, `blow_down` 15.
- Verificamos que ninguna imagen aparece en más de una partición.

### Qué significa
- El conjunto de test es representativo de todas las etiquetas, incluidas las raras, por lo que
  las comparaciones entre funciones de pérdida serán justas.
- El test **no se usará para elegir hiperparámetros ni umbrales**: eso se hará solo con validación,
  para evitar fuga de información (requisito R4).
- Con solo 15 ejemplos de test, las métricas de `conventional_mine` y `blow_down` tendrán alta
  variabilidad; por eso reportaremos media ± desviación estándar en 3 semillas.
- **Limitación:** el dataset no incluye coordenadas, y varias imágenes pueden provenir de una misma
  escena satelital. Imágenes vecinas podrían quedar en particiones distintas, lo que puede inflar
  ligeramente las métricas. Lo discutiremos en la sección V del paper.

**Archivos generados:** `splits/particion.csv`, `results/tables/tabla2_particion.csv`.

In [ ]:
# =====================================================================
# BLOQUE 3 · Caché de imágenes, métricas y línea base (R3)
# =====================================================================
# Este bloque es reejecutable: si la caché, las características o la línea base ya existen,
# las cargamos en lugar de recalcularlas.

# --- 3.1 Librerías y configuración ---
import json, time, os, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch, timm
from PIL import Image
from sklearn.metrics import fbeta_score, average_precision_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)  # evitamos llenar la salida de avisos

TRABAJO = Path('/kaggle/working')
config = json.load(open(TRABAJO / 'config_global.json'))
ETIQUETAS, RARAS = config['ETIQUETAS'], config['RARAS']
DIR_IMG = Path(config['DIR_IMG'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TAM = 160   # reducimos las imágenes de 256 a 160 píxeles para que el entrenamiento sea más rápido
config['TAM'] = TAM
json.dump(config, open(TRABAJO / 'config_global.json', 'w'), indent=1)

# Cargamos etiquetas y partición, y comprobamos que estén en el mismo orden
df_bin = pd.read_csv(TRABAJO / 'splits/etiquetas_binarias.csv')
particion = pd.read_csv(TRABAJO / 'splits/particion.csv')
assert (df_bin.image_name.values == particion.image_name.values).all(), 'Etiquetas y partición desalineadas'
Y = df_bin[ETIQUETAS].values.astype(np.float32)                               # matriz de etiquetas (40479 x 17)
IDX = {s: np.where(particion.split.values == s)[0] for s in ['train', 'val', 'test']}  # posiciones de cada partición

# --- 3.2 Caché de imágenes: decodificamos los 40.479 JPG una sola vez y los guardamos como matriz ---
# La guardamos en /tmp (no en /kaggle/working) para no inflar las salidas guardadas: son unos 3 GB
# y se reconstruye en pocos minutos en cada sesión nueva.
RUTA_CACHE = Path(f'/tmp/cache_{TAM}.npy')

def leer_imagen(nombre):
    # Abrimos el JPG, lo pasamos a RGB y lo reducimos al tamaño elegido
    with Image.open(DIR_IMG / f'{nombre}.jpg') as im:
        return np.asarray(im.convert('RGB').resize((TAM, TAM), Image.BILINEAR))

if not RUTA_CACHE.exists():
    t0 = time.time()
    tmp = Path(f'/tmp/cache_{TAM}_incompleta.npy')
    cache_tmp = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.uint8,
                                          shape=(len(df_bin), TAM, TAM, 3))
    # Leemos las imágenes en paralelo con varios hilos para acelerar
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, arr in enumerate(ex.map(leer_imagen, df_bin.image_name)):
            cache_tmp[i] = arr
    cache_tmp.flush(); del cache_tmp
    os.rename(tmp, RUTA_CACHE)   # renombramos solo al terminar, así nunca usamos una caché a medias
    print(f'Caché creada en {(time.time() - t0) / 60:.1f} min')
CACHE = np.load(RUTA_CACHE, mmap_mode='r')   # la leemos sin cargar todo de golpe en memoria
print('Caché:', CACHE.shape, CACHE.dtype)

# --- 3.3 Dataset y DataLoader para PyTorch ---
MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)   # normalización de ImageNet,
DESV = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)    # porque usamos modelos preentrenados en ImageNet

class DatasetSatelital(torch.utils.data.Dataset):
    def __init__(self, indices, aumentar):
        self.indices, self.aumentar = indices, aumentar
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        x = torch.from_numpy(np.array(CACHE[i])).permute(2, 0, 1).float().div(255)  # HWC -> CHW en [0, 1]
        if self.aumentar:
            # Aumento de datos propio de imágenes satelitales: desde el espacio no hay "arriba" ni "abajo",
            # así que volteamos y rotamos en múltiplos de 90° sin cambiar el significado de la imagen
            if torch.rand(1) < 0.5: x = x.flip(2)
            if torch.rand(1) < 0.5: x = x.flip(1)
            x = torch.rot90(x, int(torch.randint(0, 4, (1,))), dims=(1, 2))
        return (x - MEDIA) / DESV, torch.from_numpy(Y[i])

def crear_loader(indices, aumentar, batch=64, mezclar=False, semilla=0):
    g = torch.Generator(); g.manual_seed(semilla)   # fijamos la semilla del orden de los lotes
    return torch.utils.data.DataLoader(DatasetSatelital(indices, aumentar), batch_size=batch,
                                       shuffle=mezclar, num_workers=4, pin_memory=True,
                                       drop_last=mezclar, generator=g)

# --- 3.4 Métricas multietiqueta ---
# Elegimos un umbral de decisión por etiqueta usando SOLO validación (nunca test), maximizando F2,
# que es la métrica oficial de la competencia y da más peso al recall que a la precisión.
GRILLA_UMBRALES = np.round(np.arange(0.02, 0.99, 0.02), 2)

def ajustar_umbrales(P_val, Y_val):
    umbrales = np.zeros(P_val.shape[1])
    for j in range(P_val.shape[1]):
        f2 = [fbeta_score(Y_val[:, j], P_val[:, j] >= t, beta=2, zero_division=0) for t in GRILLA_UMBRALES]
        umbrales[j] = GRILLA_UMBRALES[int(np.argmax(f2))]
    return umbrales

def evaluar(P_val, Y_val, P_test, Y_test):
    # Ajustamos umbrales en validación y los aplicamos al test
    umbrales = ajustar_umbrales(P_val, Y_val)
    pred = (P_test >= umbrales).astype(int)
    por_etiqueta = pd.DataFrame({
        'umbral': umbrales,
        'f2': fbeta_score(Y_test, pred, beta=2, average=None, zero_division=0),
        'precision': precision_score(Y_test, pred, average=None, zero_division=0),
        'recall': recall_score(Y_test, pred, average=None, zero_division=0),
        'ap': average_precision_score(Y_test, P_test, average=None),   # no depende del umbral
    }, index=ETIQUETAS)
    m = {
        'f2_muestras': fbeta_score(Y_test, pred, beta=2, average='samples', zero_division=0),  # métrica de la competencia
        'f2_macro': por_etiqueta.f2.mean(),                  # promedio simple: cada etiqueta pesa igual
        'f2_macro_raras': por_etiqueta.loc[RARAS, 'f2'].mean(),
        'map_macro': por_etiqueta.ap.mean(),
        'map_raras': por_etiqueta.loc[RARAS, 'ap'].mean(),
        'f2_mineria': por_etiqueta.loc['artisinal_mine', 'f2'],
        'recall_mineria': por_etiqueta.loc['artisinal_mine', 'recall'],
        'precision_mineria': por_etiqueta.loc['artisinal_mine', 'precision'],
        'ap_mineria': por_etiqueta.loc['artisinal_mine', 'ap'],
    }
    return {k: float(v) for k, v in m.items()}, por_etiqueta

# --- 3.5 Características congeladas de ResNet-50 (para la línea base) ---
RUTA_FEATS = TRABAJO / f'results/feats_resnet50_{TAM}.npy'
if RUTA_FEATS.exists():
    FEATS = np.load(RUTA_FEATS)
else:
    t0 = time.time()
    # Cargamos ResNet-50 preentrenada SIN su capa final: devuelve un vector de 2048 características por imagen
    extractor = timm.create_model('resnet50.a1_in1k', pretrained=True, num_classes=0).to(DEVICE).eval()
    partes = []
    with torch.no_grad():
        for x, _ in crear_loader(np.arange(len(df_bin)), aumentar=False, batch=256):
            with torch.autocast('cuda', dtype=torch.float16):   # precisión mixta: más rápido en la T4
                partes.append(extractor(x.to(DEVICE, non_blocking=True)).float().cpu().numpy())
    FEATS = np.concatenate(partes).astype(np.float16)
    np.save(RUTA_FEATS, FEATS)
    del extractor; torch.cuda.empty_cache()
    print(f'Características extraídas en {(time.time() - t0) / 60:.1f} min')
print('Características:', FEATS.shape)

# --- 3.6 Línea base: regresión logística (una por etiqueta) sobre las características congeladas ---
RUTA_BASE = TRABAJO / 'results/baseline.json'
if RUTA_BASE.exists():
    base = json.load(open(RUTA_BASE))
    print('Línea base ya calculada; la cargamos.')
else:
    t0 = time.time()
    Xtr, Xva, Xte = (FEATS[IDX[s]].astype(np.float32) for s in ['train', 'val', 'test'])
    Ytr, Yva, Yte = (Y[IDX[s]] for s in ['train', 'val', 'test'])
    mejor = (-1, None, None, None)
    # Elegimos la regularización C con la mAP de validación
    # Grilla completa de C: incluye los valores que exploramos en el bloque 3b, para que una ejecución desde cero reproduzca nuestra línea base
    for C in [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.1, 1.0]:
        clf = make_pipeline(StandardScaler(),
                            OneVsRestClassifier(LogisticRegression(C=C, max_iter=500), n_jobs=-1))
        clf.fit(Xtr, Ytr)
        Pva = clf.predict_proba(Xva)
        map_val = average_precision_score(Yva, Pva, average='macro')
        print(f'  C={C}: mAP en validación = {map_val:.4f}')
        if map_val > mejor[0]:
            mejor = (map_val, C, clf, Pva)
    map_val, C, clf, Pva = mejor
    Pte = clf.predict_proba(Xte)
    metricas, por_etiqueta = evaluar(Pva, Yva, Pte, Yte)
    base = {'modelo': 'LogReg sobre ResNet-50 congelada', 'C': C, 'map_val': float(map_val),
            'tiempo_min': (time.time() - t0) / 60, **metricas}
    json.dump(base, open(RUTA_BASE, 'w'), indent=1)
    por_etiqueta.to_csv(TRABAJO / 'results/tables/baseline_por_etiqueta.csv')
    np.savez(TRABAJO / 'results/preds/baseline.npz', P_test=Pte, P_val=Pva)

# --- 3.7 Validación y resumen ---
assert CACHE.shape == (len(df_bin), TAM, TAM, 3), 'La caché no tiene la forma esperada'
assert FEATS.shape == (len(df_bin), 2048), 'Las características no tienen la forma esperada'
assert all(0 <= base[k] <= 1 for k in ['f2_muestras', 'map_macro', 'recall_mineria']), 'Métricas fuera de rango'

print('\n===== RESUMEN BLOQUE 3 =====')
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()}, indent=1))
por_etiqueta = pd.read_csv(TRABAJO / 'results/tables/baseline_por_etiqueta.csv', index_col=0)
print('\nMétricas por etiqueta rara (test):\n', por_etiqueta.loc[RARAS].round(3).to_string())
print('\nBLOQUE 3 OK ✔')

## Resultados del Bloque 3 · Línea base (R3)

### Qué hicimos
Redujimos las 40.479 imágenes a 160×160 píxeles y las guardamos en una caché para acelerar el
entrenamiento. Extrajimos con una **ResNet-50 preentrenada en ImageNet y congelada**
(`resnet50.a1_in1k`) un vector de 2.048 características por imagen, y entrenamos sobre ellas una
**regresión logística independiente por etiqueta**. Elegimos la regularización C con la mAP de
validación entre siete valores, y ajustamos un umbral de decisión por etiqueta maximizando F2 en
validación; el test solo se usó para la evaluación final.

**Métrica principal (definida antes de entrenar las redes):** mAP sobre las 6 etiquetas raras.
**Secundarias:** F2 por muestras (métrica oficial de la competencia), F2 macro de las raras y
métricas de minería artesanal (F2, recall, precisión y AP).

### Qué obtuvimos
| C | 0,0001 | 0,0003 | **0,001** | 0,003 | 0,01 | 0,1 | 1,0 |
|---|---|---|---|---|---|---|---|
| mAP validación | 0,562 | 0,584 | **0,595** | 0,592 | 0,579 | 0,549 | 0,527 |

La curva sube hasta C = 0,001 y baja hacia ambos lados: **C = 0,001 es un óptimo interior**. Con ese
valor, la línea base en test quedó así:

| Métrica (test) | Valor |
|---|---|
| F2 por muestras | 0,895 |
| F2 macro (17 etiquetas) | 0,648 |
| **F2 macro (raras)** | **0,420** |
| mAP (17 etiquetas) | 0,615 |
| **mAP (raras)** | **0,347** |

| Etiqueta rara | Umbral | F2 | Precisión | Recall | AP |
|---|---|---|---|---|---|
| artisinal_mine | 0,12 | 0,725 | 0,600 | 0,765 | 0,748 |
| conventional_mine | 0,14 | 0,608 | 0,643 | 0,600 | 0,690 |
| blooming | 0,04 | 0,440 | 0,171 | 0,725 | 0,257 |
| selective_logging | 0,04 | 0,397 | 0,188 | 0,549 | 0,266 |
| slash_burn | 0,04 | 0,240 | 0,105 | 0,355 | 0,073 |
| blow_down | 0,02 | 0,111 | 0,067 | 0,133 | 0,049 |

### Qué significa
- El F2 por muestras alto (0,895) confirma que el pipeline de datos y métricas es correcto, pero
  **oculta el problema**: está dominado por las etiquetas comunes.
- En las etiquetas raras el desempeño cae casi a la mitad (mAP 0,347 frente a 0,615). Este es el
  espacio que buscamos mejorar con fine-tuning y funciones de pérdida para el desbalance.
- La minería artesanal es la etiqueta rara mejor detectada (AP 0,748), probablemente por sus rasgos visuales distintivos (pozas y suelo removido), pero
  cuatro de cada diez alarmas son falsas.
- `slash_burn` y `blow_down` son casi indetectables con características genéricas de ImageNet.
  Con solo 15 ejemplos de test, las métricas de `blow_down` son muy inestables.

**Archivos generados:** `results/baseline.json`, `results/tables/baseline_por_etiqueta.csv`,
`results/preds/baseline.npz`, `results/feats_resnet50_160.npy`.

In [ ]:
# =====================================================================
# BLOQUE 3b · Ajustamos mejor la regularización de la línea base
# =====================================================================
# ¿Por qué hacemos esto? En el bloque 3 probamos C = 0.01, 0.1 y 1.0, y ganó 0.01, que era
# el valor más pequeño que probamos. Eso nos dejó con la duda: si hubiéramos probado valores
# todavía más pequeños, ¿habría salido algo mejor? Para no quedarnos con esa duda, ampliamos
# la búsqueda hacia abajo.
#
# Nos importa porque la línea base es nuestro punto de comparación. Si la dejamos mal ajustada,
# nuestras redes parecerían mucho mejores de lo que realmente son, y eso no sería una comparación
# justa (requisito R3).
#
# Ojo con la fuga de datos: C lo elegimos mirando SOLO la mAP de validación. El test lo usamos
# una única vez al final, con el C que gane, igual que hicimos en el bloque 3.

# Anotamos los resultados que ya teníamos del bloque 3, para no tener que repetirlos
historial = {0.001: 0.5953, 0.003: 0.5918, 0.01: 0.5792, 0.1: 0.5493, 1.0: 0.5271}

# Bajamos un paso más: probamos 0.0001 y 0.0003 para ver si la curva sigue subiendo o ya se aplanó
NUEVOS_C = [0.0001, 0.0003]

# Cargamos la línea base que guardamos en el bloque 3 para poder compararnos con ella
base_actual = json.load(open(TRABAJO / 'results/baseline.json'))

# Separamos las características y las etiquetas en entrenamiento, validación y test.
# Las características ya las calculamos en el bloque 3, así que esto es inmediato.
Xtr, Xva, Xte = (FEATS[IDX[s]].astype(np.float32) for s in ['train', 'val', 'test'])
Ytr, Yva, Yte = (Y[IDX[s]] for s in ['train', 'val', 'test'])

# Empezamos asumiendo que el mejor sigue siendo el del bloque 3; si algún valor nuevo lo supera, lo reemplazamos
mejor = (base_actual['map_val'], base_actual['C'], None, None)
t0 = time.time()

for C in NUEVOS_C:
    # Repetimos exactamente el mismo procedimiento del bloque 3 para que la comparación sea justa:
    # primero estandarizamos las características y luego entrenamos una regresión logística por etiqueta
    clf = make_pipeline(StandardScaler(),
                        OneVsRestClassifier(LogisticRegression(C=C, max_iter=500), n_jobs=-1))
    clf.fit(Xtr, Ytr)

    # Vemos qué tan bien le va en validación con la mAP (promedio de la AP de las 17 etiquetas)
    Pva = clf.predict_proba(Xva)
    map_val = average_precision_score(Yva, Pva, average='macro')
    historial[C] = round(float(map_val), 4)
    print(f'  C={C}: mAP en validación = {map_val:.4f}')

    # Si este C le gana al mejor que teníamos, nos quedamos con él
    if map_val > mejor[0]:
        mejor = (map_val, C, clf, Pva)

# Guardamos todos los valores de C que probamos con su resultado.
# Esto nos sirve en el paper para mostrar cómo elegimos C, y por si el jurado nos pregunta.
json.dump({str(k): v for k, v in sorted(historial.items())},
          open(TRABAJO / 'results/tables/baseline_busqueda_C.json', 'w'), indent=1)

if mejor[2] is not None:
    # Caso 1: algún C nuevo fue mejor en validación.
    # Recién ahora evaluamos en test (una sola vez) y reemplazamos los archivos de la línea base.
    map_val, C, clf, Pva = mejor
    Pte = clf.predict_proba(Xte)
    metricas, por_etiqueta = evaluar(Pva, Yva, Pte, Yte)   # la misma función de métricas del bloque 3
    base = {'modelo': 'LogReg sobre ResNet-50 congelada', 'C': C, 'map_val': float(map_val),
            'tiempo_min': (time.time() - t0) / 60, **metricas}
    json.dump(base, open(TRABAJO / 'results/baseline.json', 'w'), indent=1)
    por_etiqueta.to_csv(TRABAJO / 'results/tables/baseline_por_etiqueta.csv')
    np.savez(TRABAJO / 'results/preds/baseline.npz', P_test=Pte, P_val=Pva)
    print(f'\nLínea base ACTUALIZADA: nuevo C = {C}')
else:
    # Caso 2: ninguno de los valores nuevos mejoró. Nos quedamos con la línea base del bloque 3 tal cual.
    base = base_actual
    print(f"\nLínea base SIN CAMBIOS: C = {base_actual['C']} sigue siendo el mejor en validación")

# Revisamos si el ganador vuelve a ser el más pequeño de todos los que probamos.
# Si pasa eso, tendríamos que avisarlo, porque otra vez quedaría la duda de si hay algo mejor más abajo.
if base['C'] == min(historial):
    print('Aviso: el mejor C está otra vez en el borde inferior de la grilla.')

print('\n===== RESUMEN BLOQUE 3b =====')
print('mAP de validación por C:', {k: historial[k] for k in sorted(historial)})
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()}, indent=1))
print('\nBLOQUE 3b OK ✔')

## Resultados del Bloque 3b · Cómo llegamos a la grilla de C

### Qué hicimos
Durante el desarrollo, la primera búsqueda de C solo incluía 0,01, 0,1 y 1,0, y ganó 0,01, el valor
más pequeño. Como el óptimo estaba en el borde de la grilla, ampliamos la búsqueda en dos rondas:
primero 0,001 y 0,003, y luego, como 0,001 volvió a quedar en el borde, 0,0001 y 0,0003. Con eso
encontramos un óptimo interior en C = 0,001 (ver la tabla del bloque 3). Siempre elegimos C solo con
la mAP de validación, y evaluamos el test una única vez, con el valor ganador.

Para que una ejecución desde cero reprodujera esta línea base, incorporamos la grilla completa de siete
valores directamente en el bloque 3. Por eso, en esta versión, la celda 3b solo vuelve a probar 0,0001
y 0,0003 y confirma que **la línea base no cambia** (C = 0,001).

### Qué significa
- Afinar C mejoró la línea base respecto de C = 0,01 en casi todas las métricas (por ejemplo, el F2
  por muestras pasó de 0,887 a 0,895 y la precisión en minería de 0,488 a 0,600), lo que hace la
  comparación con nuestras redes más exigente y más justa.
- La **mAP de las raras no cambió** (0,347). Para las clases raras, el límite no está en cómo ajustamos
  el clasificador, sino en las **características congeladas de ImageNet**. Es un argumento a favor
  del siguiente paso: entrenar la red completa y usar funciones de pérdida pensadas para el desbalance.

**Archivos generados:** `results/tables/baseline_busqueda_C.json`.

In [ ]:
# Revisamos qué hay realmente en nuestra carpeta de trabajo, sin depender del panel Output
from pathlib import Path

TRABAJO = Path('/kaggle/working')
archivos = sorted(p for p in TRABAJO.rglob('*') if p.is_file())
print(f'Archivos en /kaggle/working: {len(archivos)}')
for p in archivos:
    print(f'  {p.relative_to(TRABAJO)}  ({p.stat().st_size / 1e6:.2f} MB)')

In [ ]:
# =====================================================================
# BLOQUE 4a · Las cuatro funciones de pérdida que vamos a comparar
# =====================================================================
# Esta es la pregunta central del proyecto: ¿qué función de pérdida ayuda más a detectar las
# etiquetas raras? Comparamos cuatro, de la más simple a la más reciente. Todas reciben los
# "logits" (la salida cruda de la red, antes de la sigmoide) y las etiquetas reales (0 o 1).
#
# Ojo: para que la comparación sea justa, las cuatro promedian la pérdida sobre todas las
# etiquetas de todas las imágenes del lote, así quedan en una escala parecida.

import torch.nn as nn
import torch.nn.functional as F

# --- 1) BCE: entropía cruzada binaria (nuestro punto de partida) ---
# Fórmula, para cada etiqueta: L = -[ y·log(p) + (1-y)·log(1-p) ],  con p = sigmoide(logit)
# Trata a todas las etiquetas por igual. Como la mayoría de los ejemplos son negativos
# (por ejemplo, el 99,2 % de las imágenes NO tienen mina), la red puede aprender a decir
# casi siempre "no" y aun así tener una pérdida baja. Ese es el problema que queremos atacar.
class PerdidaBCE(nn.Module):
    def forward(self, logits, y):
        return F.binary_cross_entropy_with_logits(logits.float(), y)

# --- 2) BCE ponderada: le damos más peso a los positivos de las etiquetas raras ---
# Fórmula: L = -[ w·y·log(p) + (1-y)·log(1-p) ],  con un peso w distinto para cada etiqueta.
# El peso "clásico" sería w = negativos / positivos, pero para blow_down eso da más de 400 y el
# entrenamiento se vuelve inestable. Por eso usamos la raíz cuadrada: w = sqrt(negativos / positivos).
# Así las raras pesan más, pero sin exagerar.
class PerdidaBCEPonderada(nn.Module):
    def __init__(self, Y_train):
        super().__init__()
        positivos = Y_train.sum(0)                          # cuántos positivos tiene cada etiqueta en train
        negativos = len(Y_train) - positivos
        peso = np.sqrt(negativos / np.maximum(positivos, 1))
        self.register_buffer('peso', torch.tensor(peso, dtype=torch.float32))  # viaja a la GPU con el modelo
    def forward(self, logits, y):
        return F.binary_cross_entropy_with_logits(logits.float(), y, pos_weight=self.peso)

# --- 3) Focal loss (Lin et al., 2017): bajamos el peso de los ejemplos "fáciles" ---
# Fórmula: L = -α_t · (1 - p_t)^γ · log(p_t)
#   donde p_t = p si y = 1, y p_t = 1 - p si y = 0 (es la probabilidad de la respuesta correcta).
# La idea: si la red ya acierta con mucha seguridad (p_t cerca de 1), el factor (1 - p_t)^γ
# se vuelve casi 0 y ese ejemplo casi no aporta. Así la red se concentra en los ejemplos
# difíciles. Usamos los valores del paper original: γ = 2 y α = 0,25.
class PerdidaFocal(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha
    def forward(self, logits, y):
        logits = logits.float()
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none')  # = -log(p_t)
        p = torch.sigmoid(logits)
        p_t = p * y + (1 - p) * (1 - y)
        perdida = (1 - p_t) ** self.gamma * bce
        if self.alpha is not None:
            alpha_t = self.alpha * y + (1 - self.alpha) * (1 - y)
            perdida = alpha_t * perdida
        return perdida.mean()

# --- 4) Asymmetric Loss, ASL (Ridnik et al., 2021): trata distinto a positivos y negativos ---
# Fórmulas:
#   para positivos (y = 1): L+ = -(1 - p)^γ+ · log(p)
#   para negativos (y = 0): L- = -(p_m)^γ- · log(1 - p_m),  con p_m = max(p - m, 0)
# Es la técnica más reciente que probamos y fue pensada justo para clasificación multietiqueta,
# donde los negativos son muchísimos más que los positivos. Hace dos cosas:
#   - Con γ- > γ+, "apaga" con más fuerza a los negativos fáciles que a los positivos.
#   - Con el margen m, si la red ya da una probabilidad muy baja a un negativo (menor que m),
#     ese negativo directamente deja de aportar a la pérdida.
# Usamos los valores recomendados por los autores: γ+ = 0, γ- = 4 y m = 0,05.
class PerdidaASL(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, margen=0.05, eps=1e-8):
        super().__init__()
        self.gamma_pos, self.gamma_neg, self.margen, self.eps = gamma_pos, gamma_neg, margen, eps
    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_m = (p - self.margen).clamp(min=0)                # probabilidad "desplazada" para los negativos
        perdida_pos = y * (1 - p) ** self.gamma_pos * torch.log(p.clamp(min=self.eps))
        perdida_neg = (1 - y) * p_m ** self.gamma_neg * torch.log((1 - p_m).clamp(min=self.eps))
        return -(perdida_pos + perdida_neg).mean()

# --- Función para crear la pérdida a partir de su nombre (la usaremos en el bloque 4b) ---
def crear_perdida(nombre):
    if nombre == 'bce':      return PerdidaBCE()
    if nombre == 'bce_pond': return PerdidaBCEPonderada(Y[IDX['train']])   # pesos calculados SOLO con train
    if nombre == 'focal':    return PerdidaFocal()
    if nombre == 'asl':      return PerdidaASL()
    raise ValueError(f'Pérdida desconocida: {nombre}')

PERDIDAS = ['bce', 'bce_pond', 'focal', 'asl']

# =====================================================================
# Validación: comprobamos que nuestras implementaciones sean correctas
# =====================================================================
# Truco para verificar: si "apagamos" los parámetros especiales, focal y ASL deben dar
# EXACTAMENTE lo mismo que la BCE normal. Si no coincidieran, habríamos programado mal la fórmula.
torch.manual_seed(0)
logits_prueba = torch.randn(64, len(ETIQUETAS)) * 3                    # logits inventados
y_prueba = (torch.rand(64, len(ETIQUETAS)) < 0.2).float()             # etiquetas inventadas

bce_ref = PerdidaBCE()(logits_prueba, y_prueba)
focal_sin_nada = PerdidaFocal(gamma=0.0, alpha=None)(logits_prueba, y_prueba)       # focal con γ = 0 y sin α
asl_sin_nada = PerdidaASL(gamma_pos=0.0, gamma_neg=0.0, margen=0.0)(logits_prueba, y_prueba)  # ASL apagada

assert torch.allclose(bce_ref, focal_sin_nada, atol=1e-5), 'Focal con γ=0 debería ser igual a BCE'
assert torch.allclose(bce_ref, asl_sin_nada, atol=1e-4), 'ASL apagada debería ser igual a BCE'
print(f'BCE = {bce_ref:.5f} | Focal(γ=0) = {focal_sin_nada:.5f} | ASL(γ=0, m=0) = {asl_sin_nada:.5f}  → coinciden ✔')

# Además, revisamos que las cuatro pérdidas den números finitos y positivos con datos de prueba
for nombre in PERDIDAS:
    valor = crear_perdida(nombre)(logits_prueba, y_prueba)
    assert torch.isfinite(valor) and valor > 0, f'La pérdida {nombre} dio un valor raro: {valor}'
    print(f'  {nombre:9s} → {valor:.5f}')

# Mostramos los pesos de la BCE ponderada para las etiquetas raras (sirven para la tabla del paper)
pesos = crear_perdida('bce_pond').peso.numpy()
print('\nPesos de la BCE ponderada (etiquetas raras):')
for e in RARAS:
    print(f'  {e:18s} w = {pesos[ETIQUETAS.index(e)]:.1f}')

print('\nBLOQUE 4a OK ✔')

## Resultados del Bloque 4a · Funciones de pérdida

### Qué hicimos
Implementamos las cuatro funciones de pérdida que vamos a comparar. Todas reciben los logits de
la red y las etiquetas reales, y promedian sobre todas las etiquetas del lote, con p = σ(logit):

| Pérdida | Fórmula (por etiqueta) | Idea principal |
|---|---|---|
| BCE | −[y·log p + (1−y)·log(1−p)] | Trata todos los ejemplos por igual (referencia) |
| BCE ponderada | −[w·y·log p + (1−y)·log(1−p)], con w = √(neg/pos) | Da más peso a los positivos de las etiquetas raras |
| Focal loss (Lin et al., 2017) | −α_t·(1−p_t)^γ·log p_t, con γ = 2 y α = 0,25 | Reduce el peso de los ejemplos fáciles |
| ASL (Ridnik et al., 2021) | Positivos: −(1−p)^γ+·log p · Negativos: −p_m^γ−·log(1−p_m), con p_m = max(p−m, 0), γ+ = 0, γ− = 4, m = 0,05 | Trata asimétricamente a positivos y negativos, y descarta los negativos muy fáciles |

Usamos la raíz cuadrada en los pesos de la BCE ponderada porque el cociente directo superaba 400
en `blow_down` y volvía inestable el entrenamiento. Los pesos se calcularon solo con train.

### Qué obtuvimos
- **Verificación de correctitud:** con datos de prueba, focal con γ = 0 y sin α, y ASL con
  γ+ = γ− = 0 y m = 0, dieron exactamente el mismo valor que la BCE (1,47052). Esto confirma que
  las fórmulas están bien implementadas.
- Pesos de la BCE ponderada para las etiquetas raras: `selective_logging` 10,9, `artisinal_mine` 10,9,
  `blooming` 11,0, `slash_burn` 13,8, `conventional_mine` 20,1 y `blow_down` 20,2.

### Qué significa
- Las cuatro pérdidas tienen escalas distintas, pero usaremos el optimizador AdamW, que normaliza
  el tamaño de los gradientes. Por eso podemos usar la misma tasa de aprendizaje en todas y
  comparar en igualdad de condiciones.
- Los hiperparámetros de focal loss y ASL son los recomendados por sus autores; no los ajustamos,
  para no favorecer a ninguna pérdida con una búsqueda extra.

In [ ]:
# =====================================================================
# BLOQUE 4b · Función de entrenamiento (todavía no entrenamos nada)
# =====================================================================
# Aquí armamos la "receta" completa para entrenar un modelo. La usaremos igual para todos los
# experimentos, cambiando solo tres cosas: la arquitectura, la función de pérdida y la semilla.
# Así nos aseguramos de que la comparación sea justa: todo lo demás es idéntico.

import math, copy

# --- Configuración del entrenamiento (la misma para todos los experimentos) ---
# La guardamos en un archivo dentro de configs/, porque las bases piden tener los
# hiperparámetros de cada experimento en el repositorio.
CONFIG_ENT = {
    'epocas': 6,                 # vueltas completas sobre train; lo revisaremos con la prueba de la 4c
    'batch': 64,                 # imágenes por lote
    'lr': {'resnet50': 2e-4,     # tasa de aprendizaje: valores típicos de fine-tuning con AdamW.
           'vit': 1e-4},         # El ViT suele necesitar una tasa más baja que la CNN para no "romperse".
    'weight_decay': 0.05,        # regularización de AdamW, para no sobreajustar
    'epocas_calentamiento': 1,   # la primera época sube la tasa poco a poco (warmup) y luego baja en coseno
    'tam_imagen': TAM,           # 160 píxeles, igual que la caché
    'seleccion': 'mAP de etiquetas raras en validación',  # cómo elegimos la mejor época
}
(TRABAJO / 'configs').mkdir(exist_ok=True)
json.dump(CONFIG_ENT, open(TRABAJO / 'configs/config_entrenamiento.json', 'w'), indent=1)

# Los dos modelos preentrenados que usamos (de la librería timm):
#   - ResNet-50: CNN clásica del curso, unos 25 millones de parámetros.
#   - DeiT-Small: Vision Transformer de tamaño parecido (unos 22 millones), para comparar
#     una CNN con un Transformer en igualdad de condiciones.
MODELOS = {'resnet50': 'resnet50.a1_in1k', 'vit': 'deit_small_patch16_224.fb_in1k'}

# Carpeta donde guardamos un archivo por experimento terminado. Si Kaggle se corta, al volver
# a ejecutar sabremos qué experimentos ya están hechos y no los repetiremos.
DIR_RUNS = TRABAJO / 'results/runs'
DIR_RUNS.mkdir(parents=True, exist_ok=True)

IDX_RARAS = [ETIQUETAS.index(e) for e in RARAS]   # posiciones de las etiquetas raras en la matriz Y


def fijar_semilla(semilla):
    # Fijamos todas las fuentes de azar para que cada experimento se pueda repetir
    random_state = np.random.RandomState(semilla)
    np.random.seed(semilla); torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    import random; random.seed(semilla)
    # cudnn.benchmark acelera las convoluciones; a cambio, los resultados pueden variar muy poco
    # entre ejecuciones. Por eso usamos 3 semillas y reportamos media ± desviación estándar.
    torch.backends.cudnn.benchmark = True


def crear_modelo(arquitectura):
    # Cargamos el modelo preentrenado en ImageNet y cambiamos su última capa por una de 17 salidas,
    # una por etiqueta. Al ViT le indicamos que las imágenes son de 160 píxeles: timm adapta
    # automáticamente sus "embeddings de posición", que vienen preparados para 224 píxeles.
    extra = {'img_size': TAM} if arquitectura == 'vit' else {}
    return timm.create_model(MODELOS[arquitectura], pretrained=True, num_classes=len(ETIQUETAS), **extra)


@torch.no_grad()
def predecir(modelo, loader):
    # Pasamos todas las imágenes por el modelo y devolvemos probabilidades (sigmoide) entre 0 y 1
    modelo.eval()
    salidas = []
    for x, _ in loader:
        with torch.autocast('cuda', dtype=torch.float16):
            salidas.append(torch.sigmoid(modelo(x.to(DEVICE, non_blocking=True)).float()).cpu())
    return torch.cat(salidas).numpy()


def map_raras(P, Y_real):
    # mAP solo sobre las 6 etiquetas raras: es nuestra métrica principal
    return float(average_precision_score(Y_real[:, IDX_RARAS], P[:, IDX_RARAS], average='macro'))


def entrenar(arquitectura, perdida, semilla, epocas=None, guardar=True):
    """Entrena un modelo completo y devuelve sus métricas en test.
    Pasos: entrenamos varias épocas, en cada una medimos la mAP de raras en validación,
    nos quedamos con la mejor época y recién ahí evaluamos UNA vez en test."""
    epocas = epocas or CONFIG_ENT['epocas']
    id_exp = f'{arquitectura}_{perdida}_s{semilla}'
    fijar_semilla(semilla)

    # Cargadores de datos: solo train tiene aumento de datos y orden aleatorio
    dl_train = crear_loader(IDX['train'], aumentar=True, batch=CONFIG_ENT['batch'], mezclar=True, semilla=semilla)
    dl_val = crear_loader(IDX['val'], aumentar=False, batch=256)
    dl_test = crear_loader(IDX['test'], aumentar=False, batch=256)
    Y_val, Y_test = Y[IDX['val']], Y[IDX['test']]

    modelo = crear_modelo(arquitectura).to(DEVICE)
    criterio = crear_perdida(perdida).to(DEVICE)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=CONFIG_ENT['lr'][arquitectura],
                                    weight_decay=CONFIG_ENT['weight_decay'])

    # Programamos la tasa de aprendizaje: sube durante el calentamiento y luego baja como un coseno
    pasos_totales = epocas * len(dl_train)
    pasos_calent = CONFIG_ENT['epocas_calentamiento'] * len(dl_train)
    def factor_lr(paso):
        if paso < pasos_calent:
            return (paso + 1) / pasos_calent
        progreso = (paso - pasos_calent) / max(1, pasos_totales - pasos_calent)
        return 0.5 * (1 + math.cos(math.pi * progreso))
    planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)

    # Precisión mixta: hacemos los cálculos en float16 para que la T4 vaya más rápido,
    # y el GradScaler evita que los gradientes pequeños se vuelvan cero
    escalador = torch.amp.GradScaler('cuda')

    mejor_map, mejor_estado, registro = -1.0, None, []
    t_inicio = time.time()
    for epoca in range(1, epocas + 1):
        t_ep = time.time()
        modelo.train()
        suma_perdida, n = 0.0, 0
        for x, y in dl_train:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = modelo(x)
            loss = criterio(logits, y)              # la pérdida la calculamos en float32 (ver bloque 4a)
            optimizador.zero_grad(set_to_none=True)
            escalador.scale(loss).backward()
            escalador.step(optimizador)
            escalador.update()
            planificador.step()
            suma_perdida += loss.item() * len(y); n += len(y)

        # Al final de cada época medimos en validación (nunca en test)
        P_val = predecir(modelo, dl_val)
        m_val_raras = map_raras(P_val, Y_val)
        m_val_todas = float(average_precision_score(Y_val, P_val, average='macro'))
        registro.append({'epoca': epoca, 'perdida_train': suma_perdida / n,
                         'map_val': m_val_todas, 'map_val_raras': m_val_raras,
                         'minutos': (time.time() - t_ep) / 60})
        print(f'  [{id_exp}] época {epoca}/{epocas} | pérdida {suma_perdida / n:.4f} | '
              f'mAP val {m_val_todas:.4f} | mAP val raras {m_val_raras:.4f} | {(time.time() - t_ep) / 60:.1f} min')

        # Si esta época es la mejor hasta ahora en validación, guardamos una copia de sus pesos
        if m_val_raras > mejor_map:
            mejor_map = m_val_raras
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
            mejor_epoca = epoca

    minutos_entrenamiento = (time.time() - t_inicio) / 60

    # Recuperamos la mejor época y recién ahora evaluamos en test, una sola vez
    modelo.load_state_dict(mejor_estado)
    P_val, P_test = predecir(modelo, dl_val), predecir(modelo, dl_test)
    metricas, por_etiqueta = evaluar(P_val, Y_val, P_test, Y_test)   # umbrales elegidos en validación

    resultado = {'id': id_exp, 'arquitectura': arquitectura, 'perdida': perdida, 'semilla': semilla,
                 'epocas': epocas, 'mejor_epoca': mejor_epoca, 'map_val_raras': mejor_map,
                 'minutos_entrenamiento': minutos_entrenamiento,
                 'parametros_M': sum(p.numel() for p in modelo.parameters()) / 1e6,
                 **metricas}

    if guardar:
        # Guardamos todo lo necesario para las tablas, las figuras y el Grad-CAM, sin volver a entrenar
        json.dump(resultado, open(DIR_RUNS / f'{id_exp}.json', 'w'), indent=1)
        json.dump(registro, open(TRABAJO / f'results/logs/{id_exp}.json', 'w'), indent=1)
        por_etiqueta.to_csv(TRABAJO / f'results/tables/por_etiqueta_{id_exp}.csv')
        np.savez(TRABAJO / f'results/preds/{id_exp}.npz', P_val=P_val, P_test=P_test)
        if semilla == 0:
            # Los pesos ocupan unos 100 MB por modelo; guardamos solo los de la semilla 0,
            # que son los que usaremos para el Grad-CAM y para el notebook de demostración
            torch.save(mejor_estado, TRABAJO / f'weights/{id_exp}.pt')

    del modelo, optimizador; torch.cuda.empty_cache()   # liberamos la memoria de la GPU para el siguiente
    return resultado


# =====================================================================
# Validación: comprobamos que los dos modelos se crean bien (sin entrenar)
# =====================================================================
x_prueba = torch.randn(2, 3, TAM, TAM, device=DEVICE)   # dos "imágenes" inventadas de 160x160
for arq in MODELOS:
    m = crear_modelo(arq).to(DEVICE).eval()
    with torch.no_grad():
        salida = m(x_prueba)
    n_param = sum(p.numel() for p in m.parameters()) / 1e6
    assert salida.shape == (2, len(ETIQUETAS)), f'{arq}: la salida debería ser (2, 17) y es {tuple(salida.shape)}'
    print(f'{arq:9s} ({MODELOS[arq]}) → salida {tuple(salida.shape)} | {n_param:.1f} M parámetros ✔')
    del m
torch.cuda.empty_cache()

print('\nConfiguración guardada en configs/config_entrenamiento.json')
print('BLOQUE 4b OK ✔')

## Resultados del Bloque 4b · Función de entrenamiento

### Qué hicimos
Definimos una única función de entrenamiento que usaremos en todos los experimentos, cambiando
solo la arquitectura, la función de pérdida y la semilla. Todo lo demás es idéntico, para que la
comparación sea controlada.

**Configuración común** (guardada en `configs/config_entrenamiento.json`):
- Optimizador AdamW con weight decay de 0,05.
- Tasa de aprendizaje de 2·10⁻⁴ para ResNet-50 y 1·10⁻⁴ para el ViT, con una época de
  calentamiento y luego decaimiento en coseno.
- Lotes de 64 imágenes de 160×160 píxeles y precisión mixta (float16) para acelerar en la T4.
- Aumento de datos con volteos y rotaciones de 90°, válidos en imágenes satelitales.
- **Selección del modelo:** nos quedamos con la época de mayor mAP de etiquetas raras en
  validación. El test se evalúa una sola vez, con esa época y con umbrales elegidos en validación.

### Qué obtuvimos
| Modelo | Nombre en timm | Parámetros |
|---|---|---|
| ResNet-50 (CNN) | `resnet50.a1_in1k` | 23,5 M |
| DeiT-Small (ViT) | `deit_small_patch16_224.fb_in1k` | 21,6 M |

Ambos modelos, preentrenados en ImageNet-1k, se adaptaron a 17 salidas (una por etiqueta) y a
imágenes de 160 píxeles; en el ViT, timm reajustó automáticamente los embeddings de posición.

### Qué significa
Los dos modelos tienen un tamaño casi igual, así que si uno supera al otro no será por tener más
parámetros, sino por su arquitectura: convoluciones frente a atención.

In [ ]:
# =====================================================================
# BLOQUE 4c · Prueba rápida: un experimento corto por modelo para medir el tiempo
# =====================================================================
# Antes de lanzar todos los experimentos, entrenamos UNA época con cada modelo para comprobar:
#   1. Que todo funciona de principio a fin (entrenar, validar y evaluar en test).
#   2. Cuánto tarda una época de verdad, para calcular si nos alcanza el tiempo.
# Usamos guardar=False para que estas pruebas NO se mezclen con los experimentos oficiales,
# y la semilla 99, que no usaremos en los experimentos reales.

minutos_por_epoca = {}
for arq in ['resnet50', 'vit']:
    t0 = time.time()
    prueba = entrenar(arq, 'bce', semilla=99, epocas=1, guardar=False)
    minutos_por_epoca[arq] = (time.time() - t0) / 60

    # Con una sola época las métricas todavía son bajas; solo revisamos que tengan sentido
    print(f'\n{arq}: {minutos_por_epoca[arq]:.1f} min (1 época + evaluación)')
    for k in ['f2_muestras', 'map_macro', 'map_raras', 'ap_mineria']:
        print(f'  {k:12s} = {prueba[k]:.4f}')
    # Validación: las métricas deben estar entre 0 y 1
    assert all(0 <= prueba[k] <= 1 for k in ['f2_muestras', 'map_macro', 'map_raras']), f'{arq}: métricas fuera de rango'

# Estimamos el tiempo total del plan completo:
#   - ResNet-50 con las 4 pérdidas y varias semillas
#   - ViT solo con la mejor pérdida y las mismas semillas
# Sumamos unos 10 minutos porque, en segundo plano, el notebook rearma la caché desde cero.
print('\nEstimación del tiempo total en segundo plano:')
for epocas in [4, 5, 6]:
    for semillas in [2, 3]:
        minutos = semillas * epocas * (4 * minutos_por_epoca['resnet50'] + minutos_por_epoca['vit']) + 10
        print(f'  {epocas} épocas x {semillas} semillas → {5 * semillas} experimentos, unas {minutos / 60:.1f} horas')

# Comprobamos que la prueba no guardó archivos oficiales (la carpeta de experimentos debe seguir vacía)
assert not list(DIR_RUNS.glob('*_s99.json')), 'La prueba no debería haber guardado resultados'
print('\nBLOQUE 4c OK ✔')

## Resultados del Bloque 4c · Prueba rápida y elección de la configuración

### Qué hicimos
Antes de lanzar todos los experimentos, entrenamos una sola época con cada modelo (ResNet-50 y
ViT, ambos con BCE y la semilla 99) para comprobar que todo funciona de principio a fin y para
medir el tiempo real de entrenamiento. Estas pruebas no se guardan como resultados oficiales.

### Qué obtuvimos (medido en la sesión de desarrollo)
| Modelo | Minutos por época | mAP raras (val, 1 época) | AP minería (test, 1 época) |
|---|---|---|---|
| ResNet-50 | 1,2 | 0,070 | 0,351 |
| ViT (DeiT-S) | 0,8 | 0,179 | 0,851 |

Con estos tiempos estimamos el costo del plan completo (12 experimentos de ResNet-50 y 3 del ViT):

| Épocas | 2 semillas | 3 semillas |
|---|---|---|
| 4 | 1,0 h | 1,5 h |
| 5 | 1,2 h | 1,8 h |
| **6** | 1,5 h | **2,1 h** |

Los tiempos varían según la máquina que asigne Kaggle: en la ejecución oficial (versión 4), cada
experimento tardó 7,4 min con ResNet-50 y 5,3 min con el ViT, algo más que en la sesión de desarrollo.

### Qué significa
- El pipeline funciona de punta a punta en ambos modelos.
- Las métricas de una sola época no son comparables con las finales: la primera época es de
  calentamiento, con una tasa de aprendizaje que todavía está subiendo. Que el ViT arranque más
  rápido es un dato curioso, pero no lo usamos para tomar ninguna decisión.
- **Configuración elegida: 6 épocas y 3 semillas.** Como los experimentos corren sin supervisión,
  preferimos no recortar épocas ni semillas.

In [ ]:
# Corrección: la persistencia de Kaggle no guarda carpetas vacías, así que volvemos a crear todas
# las carpetas que usamos y borramos los restos de la prueba que falló a mitad de camino
for sub in ['splits', 'results/tables', 'results/figures', 'results/logs', 'results/preds',
            'results/runs', 'weights', 'configs']:
    (TRABAJO / sub).mkdir(parents=True, exist_ok=True)

restos = list(TRABAJO.glob('results/**/*_s99*'))   # cualquier archivo de la prueba con semilla 99
for p in restos:
    p.unlink()
print('Carpetas creadas ✔ | archivos de prueba borrados:', [p.name for p in restos])

In [ ]:
# Revisamos qué hay en la carpeta de experimentos, para asegurarnos de que no quedó ningún resto de la prueba
print('Archivos en results/runs:', sorted(p.name for p in DIR_RUNS.iterdir()))
print('Cualquier archivo con s99 en /kaggle/working:', [str(p.relative_to(TRABAJO)) for p in TRABAJO.rglob('*s99*')])

In [ ]:
# =====================================================================
# BLOQUE 4d · Todos los experimentos (requisito R4)
# =====================================================================
# Plan:
#   Fase 1: ResNet-50 con las 4 pérdidas y 3 semillas → 12 experimentos.
#   Fase 2: elegimos la mejor pérdida mirando SOLO la validación y entrenamos el ViT con ella
#           y las mismas 3 semillas → 3 experimentos.
# Cada experimento terminado queda guardado en results/runs/. Si Kaggle se corta, al volver a
# ejecutar esta celda se saltan los que ya están hechos y solo se corren los que faltan.

import glob

SEMILLAS = [0, 1, 2]

# --- Prueba de guardado (1 época, unos 1,5 min) ---
# En la 4c probamos el entrenamiento, pero con guardar=False. Antes de lanzar 2 horas de
# experimentos, comprobamos que la parte de guardar archivos también funciona. Si algo fallara
# aquí, nos enteramos ahora y no después de horas de cómputo. Al final borramos estos archivos de prueba.
entrenar('resnet50', 'bce', semilla=99, epocas=1, guardar=True)
archivos_prueba = [p for patron in ['results/runs/*_s99.json', 'results/logs/*_s99.json',
                                    'results/tables/por_etiqueta_*_s99.csv', 'results/preds/*_s99.npz']
                   for p in glob.glob(str(TRABAJO / patron))]
assert len(archivos_prueba) == 4, f'El guardado no funcionó bien: se encontraron {len(archivos_prueba)} de 4 archivos'
for p in archivos_prueba:
    os.remove(p)
print('Prueba de guardado OK: se generaron los 4 archivos esperados y los borramos ✔\n')


def correr(arquitectura, perdida, semilla):
    # Ejecutamos un experimento, salvo que ya esté guardado de una ejecución anterior
    id_exp = f'{arquitectura}_{perdida}_s{semilla}'
    ruta = DIR_RUNS / f'{id_exp}.json'
    if ruta.exists():
        print(f'⏭  {id_exp} ya estaba hecho; lo saltamos')
        return json.load(open(ruta))
    print(f'▶  Empezamos {id_exp}')
    try:
        r = entrenar(arquitectura, perdida, semilla)
        print(f'✔  {id_exp}: mAP raras test = {r["map_raras"]:.4f} | AP minería = {r["ap_mineria"]:.4f} | '
              f'{r["minutos_entrenamiento"]:.1f} min\n')
        return r
    except Exception as e:
        # Si un experimento falla, lo anotamos y seguimos con los demás, para no perder toda la noche
        print(f'✘  ERROR en {id_exp}: {e!r}\n')
        torch.cuda.empty_cache()
        return None


# --- Fase 1: ResNet-50 con las 4 pérdidas ---
# Recorremos primero las semillas y dentro las pérdidas. Así, si la ejecución se corta,
# tendremos las 4 pérdidas con la semilla 0 completas, y la comparación será posible igual.
t_total = time.time()
for semilla in SEMILLAS:
    for perdida in PERDIDAS:
        correr('resnet50', perdida, semilla)

# --- Elegimos la mejor pérdida usando SOLO validación ---
# Promediamos la mAP de raras en validación de las 3 semillas de cada pérdida.
# No miramos el test para esta decisión: eso sería hacer trampa (fuga de datos).
fase1 = [json.load(open(p)) for p in DIR_RUNS.glob('resnet50_*.json')]
promedio_val = pd.DataFrame(fase1).groupby('perdida').map_val_raras.mean().sort_values(ascending=False)
MEJOR_PERDIDA = promedio_val.index[0]
json.dump({'mejor_perdida': MEJOR_PERDIDA, 'criterio': 'mAP raras en validación (promedio de semillas)',
           'promedios': promedio_val.round(4).to_dict()},
          open(TRABAJO / 'results/seleccion_perdida.json', 'w'), indent=1)
print('mAP de raras en validación por pérdida (promedio de semillas):')
print(promedio_val.round(4).to_string())
print(f'→ Mejor pérdida según validación: {MEJOR_PERDIDA}\n')

# --- Fase 2: ViT con la mejor pérdida ---
for semilla in SEMILLAS:
    correr('vit', MEJOR_PERDIDA, semilla)

# =====================================================================
# Resumen y validación final
# =====================================================================
resultados = pd.DataFrame([json.load(open(p)) for p in sorted(DIR_RUNS.glob('*.json'))])
columnas = ['id', 'mejor_epoca', 'map_val_raras', 'map_raras', 'f2_muestras', 'f2_macro_raras',
            'ap_mineria', 'recall_mineria', 'precision_mineria', 'minutos_entrenamiento']
resultados[columnas].to_csv(TRABAJO / 'results/tables/resumen_experimentos.csv', index=False)

horas_gpu = resultados.minutos_entrenamiento.sum() / 60
print(f'\nTiempo total de esta celda: {(time.time() - t_total) / 60:.1f} min')
print(f'Horas de GPU sumando todos los entrenamientos: {horas_gpu:.2f} h (para el reporte de costo, R6)')
print('\nResumen de los experimentos:')
print(resultados[columnas].round(4).to_string(index=False))

# Validación: deben estar los 15 experimentos (12 de ResNet-50 y 3 del ViT)
assert len(resultados) == 15, f'Esperábamos 15 experimentos y hay {len(resultados)}; revisa los mensajes de ERROR'
print('\nBLOQUE 4d OK ✔')

## Resultados del Bloque 4d · Experimentos principales y ablación (R4)

### Qué hicimos
Entrenamos ResNet-50 completa (fine-tuning) con las 4 funciones de pérdida y 3 semillas (12
experimentos). Elegimos la mejor pérdida según la mAP de raras **en validación** (promedio de
semillas), sin mirar el test, y con ella entrenamos el ViT (DeiT-Small) con las mismas 3 semillas.
Todos los experimentos usaron la misma partición, 6 épocas y la misma configuración. Antes de empezar,
la celda hace una prueba de guardado de 1 época (semilla 99) y borra sus archivos.

**Selección en validación (mAP raras, promedio):** BCE ponderada 0,425 · ASL 0,369 ·
BCE 0,350 · Focal 0,337 → elegimos la **BCE ponderada**.

### Qué obtuvimos (test, media ± desviación estándar en 3 semillas)
| Modelo | Pérdida | mAP raras | AP minería | F2 por muestras |
|---|---|---|---|---|
| Línea base (LogReg) | — | 0,347 | 0,748 | 0,895 |
| ResNet-50 | Focal | 0,311 ± 0,018 | 0,854 | 0,919 |
| ResNet-50 | BCE | 0,339 ± 0,012 | 0,867 | 0,921 |
| ResNet-50 | ASL | 0,371 ± 0,017 | 0,863 | 0,921 |
| ResNet-50 | BCE ponderada | 0,427 ± 0,009 | 0,885 | 0,921 |
| ViT (DeiT-S) | BCE ponderada | **0,520 ± 0,013** | **0,915** | **0,927** |

Tiempo por experimento: 7,4 min (ResNet-50) y 5,3 min (ViT). Total de la celda: 107,9 min.
Suma de los entrenamientos: **1,74 horas de GPU T4**.

### Qué significa
- **El fine-tuning por sí solo no resuelve las clases raras:** con BCE, ResNet-50 mejora las
  métricas generales y la minería, pero su mAP de raras (0,339) no supera a la línea base (0,347).
- **La ponderación de la pérdida es lo que marca la diferencia:** la BCE ponderada mejora la mAP de
  raras en 8,8 puntos sobre BCE, y su peor semilla (0,422) supera a la mejor de cualquier otra pérdida (0,390).
- **ASL mejora poco pero de forma consistente** sobre BCE (su peor semilla, 0,359, supera a la mejor
  de BCE, 0,352). **Focal loss es la peor:** su α = 0,25 reduce el peso de los positivos, lo contrario
  de lo que necesitan las clases raras.
- **El ViT supera a ResNet-50** con la misma pérdida (+9,3 puntos en mAP de raras; su peor semilla,
  0,506, supera a la mejor de ResNet-50, 0,437) y entrena más rápido, con un número de parámetros similar.
- El orden de las pérdidas fue el mismo en validación y en test, lo que respalda el procedimiento de selección.

**Limitaciones:** el ViT solo se evaluó con la pérdida elegida en ResNet-50; la mejor época fue la 5 o
la 6 en 14 de los 15 experimentos, así que con más épocas los modelos podrían mejorar.

**Archivos generados:** `results/runs/*.json`, `results/logs/*.json`, `results/preds/*.npz`,
`results/tables/por_etiqueta_*.csv`, `results/tables/resumen_experimentos.csv`,
`results/seleccion_perdida.json`, `weights/*_s0.pt`.

In [ ]:
# =====================================================================
# BLOQUE 5a · Tablas y figuras finales para el paper
# =====================================================================
# Aquí no entrenamos nada nuevo: solo leemos los resultados que guardamos en el bloque 4d y los
# convertimos en las tablas y figuras que irán al paper y a las diapositivas.
# Todas las tablas se guardan en CSV (para revisarlas) y en LaTeX (para pegarlas en Overleaf).

import matplotlib.pyplot as plt

# --- 5a.1 Cargamos los 15 experimentos oficiales y la línea base ---
runs = pd.DataFrame([json.load(open(p)) for p in sorted(DIR_RUNS.glob('*.json'))])
runs = runs[runs.semilla.isin([0, 1, 2])]          # solo semillas oficiales, por si quedó alguna prueba
base = json.load(open(TRABAJO / 'results/baseline.json'))

# Orden en que mostraremos los modelos: de la pérdida que peor anduvo a la mejor, y al final el ViT
ORDEN = [('resnet50', 'focal'), ('resnet50', 'bce'), ('resnet50', 'asl'),
         ('resnet50', 'bce_pond'), ('vit', 'bce_pond')]
NOMBRE_ARQ = {'resnet50': 'ResNet-50', 'vit': 'ViT (DeiT-S)'}
NOMBRE_PERD = {'bce': 'BCE', 'bce_pond': 'BCE ponderada', 'focal': 'Focal', 'asl': 'ASL'}
def etiqueta_config(arq, perd):
    return f'{NOMBRE_ARQ[arq]} + {NOMBRE_PERD[perd]}'

# Métricas que reportamos (la primera es la principal, definida antes de entrenar)
METRICAS = {'map_raras': 'mAP raras', 'f2_macro_raras': 'F2 raras', 'map_macro': 'mAP 17 etiq.',
            'f2_muestras': 'F2 muestras', 'ap_mineria': 'AP minería',
            'recall_mineria': 'Recall minería', 'precision_mineria': 'Precisión minería'}

# --- 5a.2 Tabla principal: media ± desviación estándar de las 3 semillas ---
filas = [{'Modelo': 'Línea base (LogReg)', 'Pérdida': '—',
          **{nombre: f'{base[k]:.3f}' for k, nombre in METRICAS.items()}}]
resumen_num = {}   # guardamos también los números "limpios" para las figuras
for arq, perd in ORDEN:
    d = runs[(runs.arquitectura == arq) & (runs.perdida == perd)]
    filas.append({'Modelo': NOMBRE_ARQ[arq], 'Pérdida': NOMBRE_PERD[perd],
                  **{nombre: f'{d[k].mean():.3f} ± {d[k].std():.3f}' for k, nombre in METRICAS.items()}})
    resumen_num[(arq, perd)] = d
tabla3 = pd.DataFrame(filas)
tabla3.to_csv(TRABAJO / 'results/tables/tabla3_resultados_principales.csv', index=False)
with open(TRABAJO / 'results/tables/tabla3_resultados_principales.tex', 'w') as f:
    f.write(tabla3.to_latex(index=False))

# --- 5a.3 Tabla por etiqueta rara: AP promedio de las 3 semillas ---
# Así vemos en qué etiquetas raras mejora cada modelo, y no solo el promedio
ap_por_etiqueta = {'Línea base': pd.read_csv(TRABAJO / 'results/tables/baseline_por_etiqueta.csv',
                                             index_col=0).loc[RARAS, 'ap']}
for arq, perd in ORDEN:
    aps = [pd.read_csv(TRABAJO / f'results/tables/por_etiqueta_{arq}_{perd}_s{s}.csv', index_col=0).loc[RARAS, 'ap']
           for s in [0, 1, 2]]
    ap_por_etiqueta[etiqueta_config(arq, perd)] = pd.concat(aps, axis=1).mean(axis=1)
tabla4 = pd.DataFrame(ap_por_etiqueta).round(3)
tabla4.index.name = 'Etiqueta rara'
tabla4.to_csv(TRABAJO / 'results/tables/tabla4_ap_por_etiqueta_rara.csv')
with open(TRABAJO / 'results/tables/tabla4_ap_por_etiqueta_rara.tex', 'w') as f:
    f.write(tabla4.to_latex())

# --- 5a.4 Figura 2: mAP de raras por configuración, con barras de error ---
fig, ax = plt.subplots(figsize=(7, 4))
nombres = [etiqueta_config(a, p) for a, p in ORDEN]
medias = [resumen_num[c].map_raras.mean() for c in ORDEN]
desv = [resumen_num[c].map_raras.std() for c in ORDEN]
colores = ['tab:orange' if a == 'vit' else 'tab:blue' for a, _ in ORDEN]
ax.barh(nombres, medias, xerr=desv, color=colores, capsize=4)
ax.axvline(base['map_raras'], color='gray', linestyle='--', label=f'Línea base ({base["map_raras"]:.3f})')
ax.set_xlabel('mAP de etiquetas raras en test (media ± d.e., 3 semillas)')
ax.set_title('Comparación de funciones de pérdida y arquitecturas')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig(TRABAJO / 'results/figures/fig2_map_raras.png', dpi=200); plt.show()

# --- 5a.5 Figura 3: AP por etiqueta rara (línea base vs. los modelos clave) ---
# Para no saturar la figura, mostramos solo la línea base, la BCE normal, la mejor pérdida y el ViT
claves = ['Línea base', 'ResNet-50 + BCE', 'ResNet-50 + BCE ponderada', 'ViT (DeiT-S) + BCE ponderada']
fig, ax = plt.subplots(figsize=(9, 4.5))
ancho = 0.2
x = np.arange(len(RARAS))
for i, c in enumerate(claves):
    ax.bar(x + (i - 1.5) * ancho, tabla4.loc[RARAS, c], width=ancho, label=c)
ax.set_xticks(x); ax.set_xticklabels(RARAS, rotation=20)
ax.set_ylabel('AP en test (promedio de 3 semillas)'); ax.set_ylim(0, 1)
ax.set_title('Average precision por etiqueta rara')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(TRABAJO / 'results/figures/fig3_ap_por_etiqueta.png', dpi=200); plt.show()

# --- 5a.6 Figura 4: curvas de aprendizaje (mAP de raras en validación por época) ---
# Promediamos las 3 semillas y sombreamos ± 1 desviación estándar.
# Esta figura muestra que la mejor época suele ser la última: los modelos seguían mejorando.
fig, ax = plt.subplots(figsize=(7, 4))
for arq, perd in ORDEN:
    curvas = np.array([[e['map_val_raras'] for e in json.load(open(TRABAJO / f'results/logs/{arq}_{perd}_s{s}.json'))]
                       for s in [0, 1, 2]])
    epocas = np.arange(1, curvas.shape[1] + 1)
    m, s_ = curvas.mean(0), curvas.std(0)
    ax.plot(epocas, m, marker='o', label=etiqueta_config(arq, perd),
            linestyle='--' if arq == 'vit' else '-')
    ax.fill_between(epocas, m - s_, m + s_, alpha=0.15)
ax.set_xlabel('Época'); ax.set_ylabel('mAP de etiquetas raras (validación)')
ax.set_title('Curvas de aprendizaje (media de 3 semillas)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(TRABAJO / 'results/figures/fig4_curvas_aprendizaje.png', dpi=200); plt.show()

# --- 5a.7 Costo computacional (requisito R6) ---
entorno = json.load(open(TRABAJO / 'results/entorno.json'))
costo = {
    'gpu': entorno['gpu'],
    'experimentos': int(len(runs)),
    'horas_gpu_entrenamiento': round(runs.minutos_entrenamiento.sum() / 60, 2),
    'minutos_por_experimento': runs.groupby('arquitectura').minutos_entrenamiento.mean().round(2).to_dict(),
    'nota': 'Sin contar la caché de imágenes (~3-10 min), las características de la línea base (~1 min) ni las pruebas.',
}
json.dump(costo, open(TRABAJO / 'results/costo_computacional.json', 'w'), indent=1, ensure_ascii=False)

# --- 5a.8 Validación y resumen ---
assert len(runs) == 15, f'Esperábamos 15 experimentos y hay {len(runs)}'
for nombre in ['fig2_map_raras.png', 'fig3_ap_por_etiqueta.png', 'fig4_curvas_aprendizaje.png']:
    assert (TRABAJO / 'results/figures' / nombre).exists(), f'No se guardó {nombre}'

print('===== TABLA 3: resultados principales (test) =====')
print(tabla3.to_string(index=False))
print('\n===== TABLA 4: AP por etiqueta rara (test, promedio de 3 semillas) =====')
print(tabla4.to_string())
print('\n===== COSTO =====')
print(json.dumps(costo, indent=1, ensure_ascii=False))
print('\nBLOQUE 5a OK ✔')

## Resultados del Bloque 5a · Tablas y figuras finales

### Qué hicimos
Reunimos los 15 experimentos y la línea base en las tablas y figuras del artículo: resultados
principales con media ± desviación estándar (Tabla 3), AP por etiqueta rara (Tabla 4), mAP de
raras por configuración (Fig. 2), AP por etiqueta (Fig. 3) y curvas de aprendizaje (Fig. 4).

### Qué obtuvimos
**AP por etiqueta rara en test (promedio de 3 semillas):**

| Etiqueta rara | Línea base | R50 + Focal | R50 + BCE | R50 + ASL | R50 + BCE pond. | ViT + BCE pond. |
|---|---|---|---|---|---|---|
| artisinal_mine | 0,748 | 0,854 | 0,867 | 0,863 | 0,885 | **0,915** |
| conventional_mine | 0,690 | 0,364 | 0,385 | 0,576 | 0,706 | **0,839** |
| selective_logging | 0,266 | 0,373 | 0,447 | 0,367 | 0,418 | **0,451** |
| blooming | 0,257 | 0,160 | 0,195 | 0,190 | 0,227 | **0,296** |
| blow_down | 0,049 | 0,068 | 0,088 | 0,153 | 0,236 | **0,441** |
| slash_burn | 0,073 | 0,045 | 0,050 | 0,076 | 0,092 | **0,176** |

**Costo:** 15 experimentos en una GPU Tesla T4, **1,74 horas de GPU** de entrenamiento
(7,4 min por experimento con ResNet-50 y 5,3 min con el ViT).

### Qué significa
- **La métrica global oculta el problema:** el F2 por muestras queda entre 0,919 y 0,927 en todos los
  modelos ajustados, mientras que la mAP de raras va de 0,311 a 0,520. Por eso elegimos la mAP de
  raras como métrica principal.
- **El fine-tuning sin ponderar puede perjudicar a las clases raras:** `conventional_mine` cae de
  0,690 (línea base) a 0,385 con BCE y a 0,364 con focal loss. Al entrenar toda la red, esta clase casi
  no aparece en los gradientes y el modelo la "olvida". La BCE ponderada la recupera y la supera
  ligeramente (0,706).
- **La ponderación ayuda sobre todo a las etiquetas más escasas** (`conventional_mine`, `blow_down`,
  `slash_burn`), que reciben los pesos más altos. En `selective_logging`, en cambio, la BCE normal
  obtiene un valor algo mayor (0,447 frente a 0,418).
- **El ViT con BCE ponderada es el mejor en las seis etiquetas raras**, con la mayor mejora en
  `blow_down` (de 0,049 a 0,441), aunque esta cifra se basa en solo 15 ejemplos de test.
- **Minería artesanal**, la etiqueta central del proyecto, mejora de 0,748 a 0,915 de AP, con un
  recall de 0,974 y una precisión de 0,677: detectamos casi todas las minas, con algunas falsas alarmas.
- `slash_burn` sigue siendo la etiqueta más difícil (AP máximo de 0,176).

**Archivos generados:** `results/tables/tabla3_resultados_principales.{csv,tex}`,
`results/tables/tabla4_ap_por_etiqueta_rara.{csv,tex}`, `results/figures/fig2_map_raras.png`,
`results/figures/fig3_ap_por_etiqueta.png`, `results/figures/fig4_curvas_aprendizaje.png`,
`results/costo_computacional.json`.

In [ ]:
# =====================================================================
# BLOQUE 5b · ¿El modelo detecta la mina o solo el agua? (requisito R5)
# =====================================================================
# En el bloque 1 vimos que el 88 % de las imágenes con minería artesanal también tienen agua.
# Eso nos dejó una sospecha: quizás el modelo aprendió un "atajo" del tipo "si hay agua, hay mina",
# en lugar de reconocer la mina en sí. Lo revisamos de dos formas:
#   Parte 1 (con números): ¿las falsas alarmas tienen agua más de lo normal?
#   Parte 2 (visual, Grad-CAM): ¿en qué parte de la imagen se fija el modelo para decir "mina"?
# Usamos los modelos de la semilla 0 con BCE ponderada, que son los que guardamos con sus pesos.

import torch.nn.functional as F

j_mina, j_agua = ETIQUETAS.index('artisinal_mine'), ETIQUETAS.index('water')
idx_test = IDX['test']                      # las predicciones guardadas siguen este mismo orden
Y_test = Y[idx_test]
mina_real = Y_test[:, j_mina] == 1
agua_real = Y_test[:, j_agua] == 1

def cargar_predicciones(arq):
    # Leemos la probabilidad de "mina" que dio el modelo en test y su umbral (elegido en validación)
    id_exp = f'{arq}_bce_pond_s0'
    P = np.load(TRABAJO / f'results/preds/{id_exp}.npz')['P_test'][:, j_mina]
    umbral = pd.read_csv(TRABAJO / f'results/tables/por_etiqueta_{id_exp}.csv', index_col=0).loc['artisinal_mine', 'umbral']
    return P, float(umbral)

# =====================================================================
# Parte 1: análisis numérico del atajo "agua = mina"
# =====================================================================
# Si el modelo usara el atajo, esperaríamos dos señales:
#   a) Que sus falsas alarmas tengan agua mucho más seguido que una imagen cualquiera sin mina.
#   b) Que, entre las imágenes SIN mina, les dé más probabilidad de mina a las que tienen agua.
filas = {}
for arq in ['resnet50', 'vit']:
    P, umbral = cargar_predicciones(arq)
    pred = P >= umbral
    falsas = pred & ~mina_real
    sin_mina = ~mina_real
    filas[NOMBRE_ARQ[arq]] = {
        'Falsas alarmas (n)': int(falsas.sum()),
        '% con agua en falsas alarmas': 100 * agua_real[falsas].mean() if falsas.any() else np.nan,
        '% con agua en imágenes sin mina (referencia)': 100 * agua_real[sin_mina].mean(),
        'Prob. media de mina: sin mina y CON agua': P[sin_mina & agua_real].mean(),
        'Prob. media de mina: sin mina y SIN agua': P[sin_mina & ~agua_real].mean(),
        'Recall en minas con agua': pred[mina_real & agua_real].mean(),
        f'Recall en minas sin agua (n={int((mina_real & ~agua_real).sum())})': pred[mina_real & ~agua_real].mean(),
    }
tabla_atajo = pd.DataFrame(filas).round(3)
tabla_atajo.to_csv(TRABAJO / 'results/tables/tabla5_atajo_agua.csv')
print('===== ANÁLISIS DEL ATAJO "AGUA = MINA" (test, semilla 0) =====')
print(tabla_atajo.to_string())

# =====================================================================
# Parte 2: Grad-CAM (Selvaraju et al., 2017)
# =====================================================================
# Grad-CAM responde: ¿qué zonas de la imagen hicieron subir la salida "mina"?
# Idea: tomamos los mapas de activación de una capa profunda, medimos cuánto influye cada mapa en la
# salida "mina" (con los gradientes), y los sumamos ponderados. Las zonas rojas son las que más pesaron.
# Usamos el MISMO método en los dos modelos para compararlos de forma justa:
#   - ResNet-50: última capa convolucional (layer4), que da un mapa de 5x5.
#   - ViT: salida de la normalización del último bloque. Sus "tokens" son parches de 16x16 píxeles,
#     así que los reacomodamos como una grilla de 10x10 (quitando el token de clase).

class GradCAM:
    def __init__(self, modelo, capa, es_vit=False):
        self.modelo, self.es_vit = modelo, es_vit
        self.act, self.grad = None, None
        capa.register_forward_hook(self._guardar)          # guardamos las activaciones al pasar hacia adelante
    def _guardar(self, modulo, entrada, salida):
        self.act = salida
        salida.register_hook(lambda g: setattr(self, 'grad', g))   # y sus gradientes al volver hacia atrás
    def __call__(self, x, clase):
        self.modelo.zero_grad()
        logits = self.modelo(x)
        logits[0, clase].backward()                        # derivamos solo la salida "mina"
        A, G = self.act, self.grad
        if self.es_vit:
            # Quitamos el token de clase y reacomodamos los parches como una imagen de 10x10
            n_pref = self.modelo.num_prefix_tokens
            A, G = A[:, n_pref:], G[:, n_pref:]
            lado = int(A.shape[1] ** 0.5)
            A = A.reshape(1, lado, lado, -1).permute(0, 3, 1, 2)
            G = G.reshape(1, lado, lado, -1).permute(0, 3, 1, 2)
        pesos = G.mean(dim=(2, 3), keepdim=True)           # importancia de cada mapa = promedio de su gradiente
        cam = F.relu((pesos * A).sum(1, keepdim=True))     # suma ponderada; nos quedamos con lo positivo
        cam = F.interpolate(cam, size=(TAM, TAM), mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)   # escalamos a [0, 1] para dibujarlo
        return cam.detach().cpu().numpy(), torch.sigmoid(logits[0, clase]).item()

def cargar_modelo_entrenado(arq):
    # Creamos el modelo y le cargamos los pesos que entrenamos en el bloque 4d (semilla 0)
    m = crear_modelo(arq)
    m.load_state_dict(torch.load(TRABAJO / f'weights/{arq}_bce_pond_s0.pt', map_location='cpu'))
    return m.to(DEVICE).eval()

modelo_res, modelo_vit = cargar_modelo_entrenado('resnet50'), cargar_modelo_entrenado('vit')
cam_res = GradCAM(modelo_res, modelo_res.layer4[-1])
cam_vit = GradCAM(modelo_vit, modelo_vit.blocks[-1].norm1, es_vit=True)

def preparar(i):
    # Convertimos una imagen de la caché al formato que espera el modelo (igual que en el entrenamiento, sin aumento)
    x = torch.from_numpy(np.array(CACHE[i])).permute(2, 0, 1).float().div(255)
    return ((x - MEDIA) / DESV).unsqueeze(0).to(DEVICE)

# Elegimos los ejemplos según el ViT, que fue el mejor modelo:
#   3 aciertos (las minas que detectó con más seguridad),
#   3 falsas alarmas (las imágenes sin mina a las que más probabilidad de mina les dio),
#   hasta 2 minas que no detectó.
P_vit, umbral_vit = cargar_predicciones('vit')
pred_vit = P_vit >= umbral_vit
aciertos = np.where(mina_real & pred_vit)[0]
falsas = np.where(~mina_real & pred_vit)[0]
perdidas = np.where(mina_real & ~pred_vit)[0]
ejemplos = ([(k, 'Acierto') for k in aciertos[np.argsort(-P_vit[aciertos])][:3]] +
            [(k, 'Falsa alarma') for k in falsas[np.argsort(-P_vit[falsas])][:3]] +
            [(k, 'Mina no detectada') for k in perdidas[:2]])

fig, axes = plt.subplots(len(ejemplos), 3, figsize=(8, 2.7 * len(ejemplos)))
axes = np.atleast_2d(axes)
for fila, (k, tipo) in enumerate(ejemplos):
    i = idx_test[k]                                        # posición de la imagen en la caché
    imagen = np.array(CACHE[i])
    mapa_res, p_res = cam_res(preparar(i), j_mina)
    mapa_vit, p_vit = cam_vit(preparar(i), j_mina)
    assert mapa_res.shape == (TAM, TAM) and 0 <= mapa_res.min() and mapa_res.max() <= 1, 'Grad-CAM de ResNet raro'
    assert mapa_vit.shape == (TAM, TAM) and 0 <= mapa_vit.min() and mapa_vit.max() <= 1, 'Grad-CAM del ViT raro'

    axes[fila, 0].imshow(imagen)
    axes[fila, 0].set_title(f'{tipo} | agua: {"sí" if agua_real[k] else "no"}', fontsize=8)
    for col, (mapa, p, nombre) in enumerate([(mapa_res, p_res, 'ResNet-50'), (mapa_vit, p_vit, 'ViT')], start=1):
        axes[fila, col].imshow(imagen)
        axes[fila, col].imshow(mapa, cmap='jet', alpha=0.45)   # superponemos el mapa de calor
        axes[fila, col].set_title(f'Grad-CAM {nombre} | p(mina) = {p:.2f}', fontsize=8)
    for a in axes[fila]:
        a.axis('off')
plt.suptitle('Grad-CAM para "minería artesanal" (test, semilla 0, BCE ponderada)', fontsize=10)
plt.tight_layout(); plt.savefig(TRABAJO / 'results/figures/fig5_gradcam_mineria.png', dpi=200); plt.show()

# Guardamos qué imágenes mostramos, por si el jurado pregunta por alguna
pd.DataFrame([{'image_name': df_bin.image_name[idx_test[k]], 'tipo': t, 'agua': bool(agua_real[k]),
               'p_mina_vit': round(float(P_vit[k]), 3)} for k, t in ejemplos]
             ).to_csv(TRABAJO / 'results/tables/gradcam_ejemplos.csv', index=False)

assert (TRABAJO / 'results/figures/fig5_gradcam_mineria.png').exists(), 'No se guardó la figura de Grad-CAM'
print(f'\nEjemplos mostrados: {len(aciertos)} aciertos, {len(falsas)} falsas alarmas y '
      f'{len(perdidas)} minas no detectadas en total (ViT); en la figura van {len(ejemplos)}.')
print('\nBLOQUE 5b OK ✔')

## Resultados del Bloque 5b · ¿El modelo detecta la mina o solo el agua? (R5)

### Qué hicimos
Como el 88 % de las imágenes con minería artesanal también tienen agua, sospechamos que el modelo
podía haber aprendido un atajo del tipo "agua = mina". Lo revisamos de dos formas, con los modelos
de la semilla 0 entrenados con BCE ponderada:
1. **Análisis numérico** de las falsas alarmas y de las probabilidades en test.
2. **Grad-CAM** (Selvaraju et al., 2017) sobre la salida "minería artesanal", aplicado con el mismo
   método a ResNet-50 (última capa convolucional, grilla de 5×5) y al ViT (último bloque, grilla de 10×10).

### Qué obtuvimos
| Indicador (test) | ResNet-50 | ViT |
|---|---|---|
| Falsas alarmas | 19 | 26 |
| % con agua en las falsas alarmas | 52,6 % | 76,9 % |
| % con agua en imágenes sin mina (referencia) | 17,7 % | 17,7 % |
| Prob. media de mina en imágenes sin mina, con agua | 0,021 | 0,020 |
| Prob. media de mina en imágenes sin mina, sin agua | 0,005 | 0,004 |
| Recall en minas con agua (n = 47) | 0,915 | 1,000 |
| Recall en minas sin agua (n = 4) | 1,000 | 1,000 |

**Grad-CAM:** en los aciertos, ambos modelos se concentran en los parches de suelo desnudo y en
las pozas de relave dentro del bosque. En las falsas alarmas, se concentran en **bancos de arena y
sedimento claro de los ríos**. En ningún caso el cauce de agua es la zona más importante.

### Qué significa
- Los números muestran que el agua está asociada a las falsas alarmas: aparece entre 3,0 y 4,3
  veces más de lo esperado, y eleva unas 4 a 5 veces la probabilidad de mina en imágenes sin mina.
- Sin embargo, el Grad-CAM muestra que **el modelo no mira el agua, sino el sedimento claro**. La
  confusión real es entre **relaves mineros y depósitos naturales de los ríos** (bancos de arena),
  que comparten color y textura. Como esos depósitos solo aparecen junto a ríos, el agua queda
  asociada a las falsas alarmas de forma indirecta.
- Las 4 minas sin agua de test se detectaron todas, lo que confirma que el modelo no depende solo
  del agua (aunque son pocos casos).
- El ViT, que es el mejor modelo, tiene una proporción mayor de falsas alarmas con agua: un mejor
  desempeño no garantiza estar libre de sesgos.
- Algunas falsas alarmas son visualmente indistinguibles de zonas mineras y podrían ser **errores de
  etiquetado** del dataset.
- **Implicación práctica:** en un sistema real de vigilancia, las alertas cerca de ríos con bancos
  de arena deberían pasar por revisión humana antes de tomar acciones.

**Archivos generados:** `results/tables/tabla5_atajo_agua.csv`,
`results/figures/fig5_gradcam_mineria.png`, `results/tables/gradcam_ejemplos.csv`.

In [ ]:
# Juntamos todas las figuras en un solo archivo .zip para descargarlas de una vez
import shutil
shutil.make_archive(str(TRABAJO / 'figuras'), 'zip', TRABAJO / 'results/figures')
print('Listo:', sorted(p.name for p in (TRABAJO / 'results/figures').iterdir()))